Imports

In [1]:
# imports
from esdl import esdl
from esdl.esdl_handler import EnergySystemHandler
import pandas as pd
import numpy as np
import numpy_financial as npf
from decimal import Decimal, ROUND_HALF_UP
import matplotlib.pyplot as plt

In [2]:
# Import the business case runner
from set_up.generic_business_case_runner import *

In [3]:
# This imports all input variables from OWF_input_data.py
from OWF_input_data import *

# import all data from ESDL
from NSE_get_data_from_ESDL import *



Active Scenario: most_likely (Index: 1)
All variables saved to NSE_get_data_from_ESDL.pkl


In [4]:
asset_parameters

name,power,efficiency,investment_costs,fixed_opex,variable_opex,wacc
TNVDW,700000000.0,,1750.0,2.25,5.0,8.5
Electrolyzer,500.0,0.6,2000.0,2.0,0.0,8.25
Offtaker,24900000.0,,0.0,0.0,0.0,10.5


Set up calendar_year, business_case_year, operations_years, and decommissioning_years lists

In [5]:
from set_up.construct_timelines import (
    construct_calendar_year_list,
    construct_business_case_year_list,
    construct_operations_years_list,
    construct_decommissioning_years_list
)

Start business case analysis

Construction phase

In [7]:
def construction_phase(**kwargs):
    ''' This function creates a dataframe that contains all cashflows in the construction phase'''

    # 1. Get the parameters we need
    capex = kwargs['capex']
    duration_construction = kwargs['duration_construction']

    # 2. Get the timelines
    calendar_years = construct_calendar_year_list(**kwargs)
    
    # 3. Construct df
    row_names = ['capex', 'total_cashflow_investment']
    df_construction_phase = pd.DataFrame(0.0, index=row_names, columns=calendar_years)
    df_construction_phase.columns.name = "construction_phase"

    # 4. Calculate yearly CAPEX value 
    yearly_capex = -capex / duration_construction
    
    # 5. Add CAPEX numbers to construction years
    df_construction_phase.loc['capex'] = np.where(
            df_construction_phase.columns.isin(construction_years_list),
            yearly_capex, 0.0)
    
    # 6. Calculate total investments
    # at this moment only one cashflow is in construction phase, can be expanded later
    total_cashflow_investment = df_construction_phase.loc['capex']
    df_construction_phase.loc['total_cashflow_investment'] = total_cashflow_investment
   
    return df_construction_phase

In [8]:
df_construction_phase = construction_phase(**owf_parameters)
df_construction_phase.style.format(precision=2)

construction_phase,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
capex,0.00,-574.00,-574.00,-574.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
total_cashflow_investment,0.00,-574.00,-574.00,-574.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


Operational phase

In [9]:
def operational_phase(**kwargs):
    ''' This function creates a dataframe that contains all cashflows in the operational phase'''

    # 1. Get the parameters we need
    inflation = kwargs['inflation']
    opex_df = kwargs['opex_df']
    revenues_to_electrolyser = kwargs['revenues_to_electrolyser']
    revenues_to_market = kwargs['revenues_to_market']

    # 2. Get the timelines
    calendar_years = construct_calendar_year_list(**kwargs)
    operational_years = construct_operations_years_list(**kwargs)
    business_case_years = construct_business_case_year_list(**kwargs)
    
    # 3. construct df
    row_names = [
        'opex', 'total_outflow_opex', 'revenues_electricity_ppa',
        'revenues_electricity_market', 'total_revenues_opex', 
        'net_cashflow_operations'
    ]
    
    df_operational_phase = pd.DataFrame(0.0, index=row_names, columns=calendar_years)
    df_operational_phase.columns.name = "operational_phase"

    # 4. Calculate inflation factors array 
    inf_factors = (1 + inflation) ** business_case_years
    # Turn it into a series to give the calendar years as index
    # This way we can use it in our loop over the operational years
    inf_factor_series = pd.Series(inf_factors, index=calendar_years)

    # 5. Calculate the cashflows
    for x in operational_years:    

        # Get the inflation factor for that year
        inf_factor = inf_factor_series[x]

        # 6. Outflows

        # opex
        df_operational_phase.loc['opex',x] = -opex_df.loc['opex',x] * inf_factor
    
        # 7. Revenues
        
        df_operational_phase.loc['revenues_electricity_ppa',x] = revenues_to_electrolyser.loc['owf_revenues_to_electrolyser',x] * inf_factor
        df_operational_phase.loc['revenues_electricity_market',x] = revenues_to_market.loc['owf_revenues_to_market',x] * inf_factor

    # 8. Calculate totals
    outflow_rows = ['opex']
    df_operational_phase.loc['total_outflow_opex'] = df_operational_phase.loc[outflow_rows].sum()

    revenue_rows = ['revenues_electricity_ppa', 'revenues_electricity_market']
    df_operational_phase.loc['total_revenues_opex'] = df_operational_phase.loc[revenue_rows].sum()

    # Net cashflow from operation (=EBITDA)
    df_operational_phase.loc['net_cashflow_operations'] = (
        df_operational_phase.loc['total_outflow_opex'] + 
        df_operational_phase.loc['total_revenues_opex'])
    
    return df_operational_phase

In [11]:
df_operational_phase = operational_phase(**owf_parameters)
df_operational_phase.style.format(precision=2)

operational_phase,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
opex,0.00,0.00,0.00,0.00,-39.03,-38.63,-38.19,-37.73,-37.23,-36.69,-36.12,-35.51,-34.86,-34.17,-34.86,-35.56,-36.27,-36.99,-37.73,-38.49,-39.26,-40.04,-40.84,-41.66,-42.49,-43.34,-44.21,-45.09,-45.99,0.00,0.00
total_outflow_opex,0.00,0.00,0.00,0.00,-39.03,-38.63,-38.19,-37.73,-37.23,-36.69,-36.12,-35.51,-34.86,-34.17,-34.86,-35.56,-36.27,-36.99,-37.73,-38.49,-39.26,-40.04,-40.84,-41.66,-42.49,-43.34,-44.21,-45.09,-45.99,0.00,0.00
revenues_electricity_ppa,0.00,0.00,0.00,0.00,103.42,105.44,107.49,109.59,111.73,113.91,116.13,118.39,120.70,123.05,120.88,118.57,116.12,113.52,110.78,107.88,104.81,101.59,98.19,94.61,96.51,98.44,100.40,102.41,104.46,0.00,0.00
revenues_electricity_market,0.00,0.00,0.00,0.00,14.30,14.14,13.96,13.78,13.58,13.37,13.14,12.90,12.65,12.37,12.56,12.75,12.95,13.14,13.34,13.55,13.75,13.96,14.17,14.38,14.67,14.96,15.26,15.57,15.88,0.00,0.00
total_revenues_opex,0.00,0.00,0.00,0.00,117.72,119.58,121.46,123.37,125.31,127.27,129.27,131.29,133.34,135.43,133.44,131.32,129.07,126.67,124.12,121.42,118.56,115.54,112.36,108.99,111.17,113.40,115.67,117.98,120.34,0.00,0.00
net_cashflow_operations,0.00,0.00,0.00,0.00,78.69,80.95,83.26,85.64,88.08,90.58,93.15,95.78,98.48,101.25,98.58,95.77,92.80,89.68,86.39,82.94,79.31,75.50,71.51,67.34,68.68,70.06,71.46,72.89,74.34,0.00,0.00


Decommissioning phase

In [12]:
def decommissioning_phase(**kwargs):
    ''' This function creates a dataframe that contains all cashflows in the decommissioning phase'''

    # 1. Get the parameters we need
    inflation = kwargs['inflation']

    # 2. Get the timelines
    calendar_years = construct_calendar_year_list(**kwargs)
    decommissioning_years = construct_decommissioning_years_list(**kwargs)
    business_case_years = construct_business_case_year_list(**kwargs)

    # 3. construct df
    row_names = ['decommissioning_costs', 'total_cashflow_decommissioning']
    df_decommissioning_phase = pd.DataFrame(0.0, index=row_names, columns=calendar_years)
    df_decommissioning_phase.columns.name = "decommissioning_phase"

    # 4. Calculate decommissioning costs
    # Note that the capex_abex_decex is based on 2030-2040-2050 numbers, so no inflation factor is applied
    # If this is changed to a decommissioning percentage of the total capex, inflation should be added
    total_decommissioning_costs = -(df_yearly_data.loc['capex_abex_decex'] * new_windfarm_capacity / 1000) 
    yearly_decommissioning_costs = total_decommissioning_costs / len(decommissioning_years)

    # Get the costs only for the years that are decommissioning years
    costs = yearly_decommissioning_costs.loc[decommissioning_years] 

    df_decommissioning_phase.loc['decommissioning_costs', decommissioning_years] = costs

    # 5. Calculate totals
    df_decommissioning_phase.loc['total_cashflow_decommissioning'] = df_decommissioning_phase.loc['decommissioning_costs']  

    return df_decommissioning_phase

In [14]:
df_decommissioning_phase = decommissioning_phase(**owf_parameters)
df_decommissioning_phase.style.format(precision=2)

decommissioning_phase,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
decommissioning_costs,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-57.12,-57.12
total_cashflow_decommissioning,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-57.12,-57.12


Taxes & profits - part 1

In [17]:
from finances.taxes_debt_loans import taxes_and_profits_part1

df_taxes_and_profits_part1 = taxes_and_profits_part1(df_operational_phase,**owf_parameters)
df_taxes_and_profits_part1.style.format(precision=2)

taxes_and_profits,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
depreciation,0.00,0.00,0.00,0.00,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,0.00,0.00
ebit,0.00,0.00,0.00,0.00,9.81,12.07,14.38,16.76,19.20,21.70,24.27,26.90,29.60,32.37,29.70,26.89,23.92,20.80,17.51,14.06,10.43,6.62,2.63,-1.54,-0.20,1.18,2.58,4.01,5.46,0.00,0.00


Debt and loan - part 1

In [21]:
from finances.taxes_debt_loans import debt_and_loan_part1

df_debt_and_loan_part1 = debt_and_loan_part1(**owf_parameters)
df_debt_and_loan_part1.style.format(precision=2)

debt_and_loan,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
begin_of_year,0.00,0.00,430.50,861.00,1291.50,1231.65,1168.81,1102.82,1033.53,960.79,884.40,804.19,719.98,631.55,538.70,441.21,338.84,231.36,118.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
drawdown,0.00,430.50,430.50,430.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
repayment_capital,0.00,0.00,0.00,0.00,-59.85,-62.84,-65.99,-69.29,-72.75,-76.39,-80.21,-84.22,-88.43,-92.85,-97.49,-102.37,-107.48,-112.86,-118.50,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00
end_of_year,0.00,430.50,861.00,1291.50,1231.65,1168.81,1102.82,1033.53,960.79,884.40,804.19,719.98,631.55,538.70,441.21,338.84,231.36,118.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
repayment_interest,0.00,0.00,0.00,0.00,-64.58,-61.58,-58.44,-55.14,-51.68,-48.04,-44.22,-40.21,-36.00,-31.58,-26.93,-22.06,-16.94,-11.57,-5.93,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00


Taxes & profits - part 2

In [27]:
from finances.taxes_debt_loans import taxes_and_profits_part2

df_taxes_and_profits_part2 = taxes_and_profits_part2(df_operational_phase,
                                                     df_taxes_and_profits_part1,
                                                     df_debt_and_loan_part1,
                                                     **owf_parameters)
df_taxes_and_profits_part2.style.format(precision=2)

taxes_and_profits,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
depreciation,0.00,0.00,0.00,0.00,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,-68.88,0.00,0.00
ebit,0.00,0.00,0.00,0.00,9.81,12.07,14.38,16.76,19.20,21.70,24.27,26.90,29.60,32.37,29.70,26.89,23.92,20.80,17.51,14.06,10.43,6.62,2.63,-1.54,-0.20,1.18,2.58,4.01,5.46,0.00,0.00
interest_costs,0.00,0.00,0.00,0.00,-64.58,-61.58,-58.44,-55.14,-51.68,-48.04,-44.22,-40.21,-36.00,-31.58,-26.93,-22.06,-16.94,-11.57,-5.93,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00
ebt,0.00,0.00,0.00,0.00,-54.76,-49.51,-44.06,-38.38,-32.48,-26.34,-19.95,-13.31,-6.40,0.80,2.77,4.83,6.98,9.23,11.58,14.06,10.43,6.62,2.63,-1.54,-0.20,1.18,2.58,4.01,5.46,0.00,0.00
tax_expenses,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.21,-0.71,-1.25,-1.80,-2.38,-2.99,-3.63,-2.69,-1.71,-0.68,0.00,0.00,-0.30,-0.66,-1.03,-1.41,0.00,0.00
net_profits,0.00,0.00,0.00,0.00,-54.76,-49.51,-44.06,-38.38,-32.48,-26.34,-19.95,-13.31,-6.40,0.59,2.06,3.58,5.18,6.85,8.60,10.43,7.74,4.91,1.95,-1.54,-0.20,0.87,1.91,2.97,4.05,0.00,0.00


Debt & loan - part 2

In [29]:
from finances.taxes_debt_loans import debt_and_loan_part2

df_debt_and_loan_part2 = debt_and_loan_part2(df_operational_phase,
                                             df_debt_and_loan_part1,
                                             df_taxes_and_profits_part2,
                                             **owf_parameters)

df_debt_and_loan_part2.style.format(precision=2)

debt_and_loan,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
begin_of_year,0.00,0.00,430.50,861.00,1291.50,1231.65,1168.81,1102.82,1033.53,960.79,884.40,804.19,719.98,631.55,538.70,441.21,338.84,231.36,118.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
drawdown,0.00,430.50,430.50,430.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
repayment_capital,0.00,0.00,0.00,0.00,-59.85,-62.84,-65.99,-69.29,-72.75,-76.39,-80.21,-84.22,-88.43,-92.85,-97.49,-102.37,-107.48,-112.86,-118.50,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00
end_of_year,0.00,430.50,861.00,1291.50,1231.65,1168.81,1102.82,1033.53,960.79,884.40,804.19,719.98,631.55,538.70,441.21,338.84,231.36,118.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
repayment_interest,0.00,0.00,0.00,0.00,-64.58,-61.58,-58.44,-55.14,-51.68,-48.04,-44.22,-40.21,-36.00,-31.58,-26.93,-22.06,-16.94,-11.57,-5.93,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00
cash_flow_for_debt,0.00,0.00,0.00,0.00,78.69,80.95,83.26,85.64,88.08,90.58,93.15,95.78,98.48,101.05,97.87,94.52,91.00,87.30,83.40,79.31,76.62,73.79,70.83,67.34,68.68,69.75,70.79,71.85,72.93,0.00,0.00
debt_service,0.00,0.00,0.00,0.00,-124.43,-124.43,-124.43,-124.43,-124.43,-124.43,-124.43,-124.43,-124.43,-124.43,-124.43,-124.43,-124.43,-124.43,-124.43,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00
cash_after_debt_service,0.00,0.00,0.00,0.00,-45.74,-43.48,-41.16,-38.79,-36.35,-33.84,-31.28,-28.65,-25.94,-23.38,-26.56,-29.90,-33.43,-37.13,-41.03,79.31,76.62,73.79,70.83,67.34,68.68,69.75,70.79,71.85,72.93,0.00,0.00


Project reserves

In [30]:
from finances.project_reserves_and_equity import project_reserves

df_project_reserves = project_reserves(**owf_parameters)
df_project_reserves.style.format(precision=2)

project_reserves,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
contingency_injection,-172.20,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
contingency_reserve_balance,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,172.20,0.00
contingency_reserve_to_dividents,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,172.20


Equity funding

In [31]:
from finances.project_reserves_and_equity import equity_funding

df_equity_funding = equity_funding(df_decommissioning_phase,
               df_debt_and_loan_part2,
               df_construction_phase,
               **owf_parameters)

df_equity_funding.style.format(precision=2)

equity_funding,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
equity_injection,-172.20,-143.50,-143.50,-143.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-57.12,-57.12
dividents_results,0.00,0.00,0.00,0.00,-45.74,-43.48,-41.16,-38.79,-36.35,-33.84,-31.28,-28.65,-25.94,-23.38,-26.56,-29.90,-33.43,-37.13,-41.03,79.31,76.62,73.79,70.83,67.34,68.68,69.75,70.79,71.85,72.93,0.00,172.20
equity_cash_flow_result,-172.20,-143.50,-143.50,-143.50,-45.74,-43.48,-41.16,-38.79,-36.35,-33.84,-31.28,-28.65,-25.94,-23.38,-26.56,-29.90,-33.43,-37.13,-41.03,79.31,76.62,73.79,70.83,67.34,68.68,69.75,70.79,71.85,72.93,-57.12,115.08


Present value of cash flows

In [32]:
from finances.present_value_and_cumulative_cashflows import present_value_cashflows

df_present_value_cashflows = present_value_cashflows(df_construction_phase,
                                                     df_operational_phase,
                                                     df_decommissioning_phase,
                                                     df_equity_funding,
                                                     **owf_parameters)

df_present_value_cashflows.style.format(precision=2)

present_value,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
sum_net_project_cash_flows,0.00,-574.00,-574.00,-574.00,78.69,80.95,83.26,85.64,88.08,90.58,93.15,95.78,98.48,101.25,98.58,95.77,92.80,89.68,86.39,82.94,79.31,75.50,71.51,67.34,68.68,70.06,71.46,72.89,74.34,-57.12,-57.12
present_value_net_cashflows,0.00,-529.03,-487.59,-449.39,56.78,53.83,51.04,48.38,45.86,43.47,41.20,39.04,37.00,35.06,31.46,28.17,25.16,22.41,19.89,17.60,15.51,13.61,11.88,10.31,9.69,9.11,8.57,8.05,7.57,-5.36,-4.94
cumulative_value_net_cashflows,0.00,-529.03,-1016.62,-1466.01,-1409.23,-1355.39,-1304.36,-1255.98,-1210.12,-1166.65,-1125.45,-1086.41,-1049.41,-1014.35,-982.88,-954.71,-929.56,-907.15,-887.25,-869.65,-854.14,-840.53,-828.64,-818.33,-808.64,-799.52,-790.95,-782.90,-775.33,-780.69,-785.63
present_value_equity_cashflows,-172.20,-132.26,-121.90,-112.35,-33.00,-28.91,-25.23,-21.91,-18.92,-16.24,-13.83,-11.68,-9.75,-8.10,-8.48,-8.80,-9.06,-9.28,-9.45,16.83,14.99,13.30,11.77,10.31,9.69,9.07,8.49,7.94,7.43,-5.36,9.96


Cumulative equity and debt cash flows

In [33]:
from finances.present_value_and_cumulative_cashflows import cumulative_equity_and_debt_cashflows

df_cumulative_equity_and_debt_cashflows = cumulative_equity_and_debt_cashflows(df_equity_funding,
                                                                               df_debt_and_loan_part1,
                                                                               **owf_parameters)
df_cumulative_equity_and_debt_cashflows.style.format(precision=2)

cumulative equity and debt,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
cumulative_equity_cashflow,-172.20,-315.70,-459.20,-602.70,-648.44,-691.91,-733.08,-771.86,-808.21,-842.05,-873.33,-901.98,-927.92,-951.30,-977.85,-1007.76,-1041.18,-1078.31,-1119.34,-1040.03,-963.41,-889.62,-818.78,-751.45,-682.76,-613.01,-542.22,-470.37,-397.43,-454.55,-339.47
cumulative_debt_cashflow,-0.00,-430.50,-861.00,-1291.50,-1231.65,-1168.81,-1102.82,-1033.53,-960.79,-884.40,-804.19,-719.98,-631.55,-538.70,-441.21,-338.84,-231.36,-118.50,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00


Discounted cash flows for levelized cost

In [44]:
def discounted_cashflows_for_levelized_cost(df_construction_phase,
                                            df_operational_phase,
                                            df_decommissioning_phase,
                                            df_taxes_and_profits_part2,
                                            df_project_reserves,
                                            **kwargs):
    '''This function creates a dataframe that contains the discounted cashflows for levelized cost calculations'''

    # 1. Get the parameters we need
    wacc = kwargs['wacc']

    # 2. Get the timelines
    calendar_years = construct_calendar_year_list(**kwargs)
    business_case_years = construct_business_case_year_list(**kwargs)
    operational_years = construct_operations_years_list(**kwargs)

    # 3. Construct df
    row_names = [
        'capex', 'opex', 'decommissioning', 'interest_costs', 
        'contingency', 'tax_expenses', 'revenues_ppa', 
        'revenues_market', 'total_electricity_produced'
    ]
    df_discounted_levelized = pd.DataFrame(0.0, index=row_names, columns=calendar_years)
    df_discounted_levelized.columns.name = "discounted cashflows for levelized cost"

    # 4. Set up the discount term and calculate levelized costs and revenues
    discount_term = (1 + wacc) ** np.array(business_case_years)

# LEVELIZED COSTS

    df_discounted_levelized.loc['capex'] = df_construction_phase.loc['capex'] / discount_term
    df_discounted_levelized.loc['opex'] = df_operational_phase.loc['opex'] / discount_term
    df_discounted_levelized.loc['decommissioning'] = df_decommissioning_phase.loc['decommissioning_costs'] / discount_term
    df_discounted_levelized.loc['interest_costs'] = df_taxes_and_profits_part2.loc['interest_costs'] / discount_term

    # contingency (injection + divident return)
    df_discounted_levelized.loc['contingency'] = (
        (df_project_reserves.loc['contingency_injection'] 
         + df_project_reserves.loc['contingency_reserve_to_dividents']) / discount_term)

    # tax expenses
    df_discounted_levelized.loc['tax_expenses'] = df_taxes_and_profits_part2.loc['tax_expenses'] / discount_term

# LEVELIZED REVENUES

    df_discounted_levelized.loc['revenues_ppa'] = df_operational_phase.loc['revenues_electricity_ppa'] / discount_term
    df_discounted_levelized.loc['revenues_market'] = df_operational_phase.loc['revenues_electricity_market'] / discount_term

    # calculate discounted production
    production = owf_sold_electricity.loc['owf_sold_electricity', calendar_years]
    df_discounted_levelized.loc['total_electricity_produced'] = np.where(
        production.index.isin(operational_years),
        production / discount_term,
        0.0
    )


    return df_discounted_levelized



In [45]:
df_discounted_cashflows_for_levelized_cost = discounted_cashflows_for_levelized_cost(df_construction_phase,
                                                                                     df_operational_phase,
                                                                                     df_decommissioning_phase,
                                                                                     df_taxes_and_profits_part2,
                                                                                     df_project_reserves,
                                                                                     **owf_parameters)
df_discounted_cashflows_for_levelized_cost.style.format(precision=2)

discounted cashflows for levelized cost,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
capex,0.00,-529.03,-487.59,-449.39,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
opex,0.00,0.00,0.00,0.00,-28.16,-25.69,-23.41,-21.31,-19.38,-17.61,-15.98,-14.48,-13.10,-11.83,-11.12,-10.46,-9.83,-9.24,-8.69,-8.17,-7.68,-7.22,-6.79,-6.38,-6.00,-5.64,-5.30,-4.98,-4.68,0.00,0.00
decommissioning,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-5.36,-4.94
interest_costs,0.00,0.00,0.00,0.00,-46.60,-40.96,-35.82,-31.15,-26.91,-23.05,-19.56,-16.39,-13.52,-10.93,-8.60,-6.49,-4.59,-2.89,-1.36,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00
contingency,-172.20,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,14.90
tax_expenses,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.07,-0.23,-0.37,-0.49,-0.59,-0.69,-0.77,-0.53,-0.31,-0.11,0.00,0.00,-0.04,-0.08,-0.11,-0.14,0.00,0.00
revenues_ppa,0.00,0.00,0.00,0.00,74.63,70.12,65.89,61.91,58.17,54.66,51.36,48.26,45.35,42.61,38.58,34.88,31.48,28.36,25.51,22.90,20.50,18.32,16.32,14.49,13.62,12.81,12.04,11.32,10.64,0.00,0.00
revenues_market,0.00,0.00,0.00,0.00,10.32,9.40,8.56,7.78,7.07,6.41,5.81,5.26,4.75,4.28,4.01,3.75,3.51,3.28,3.07,2.88,2.69,2.52,2.35,2.20,2.07,1.95,1.83,1.72,1.62,0.00,0.00
total_electricity_produced,0.00,0.00,0.00,0.00,2582485.50,2380170.97,2193705.96,2021848.81,1863455.13,1717470.16,1582921.81,1458914.11,1344621.30,1239282.30,1142195.67,1052714.90,970244.15,894234.24,824179.02,759612.00,700103.23,645256.43,594706.39,548116.49,505176.49,465600.45,429124.84,395506.76,364522.36,0.00,0.00


Levelized costs

In [46]:
def levelized_cost_and_revenues(df_discounted_cashflows_for_levelized_cost,
                                **kwargs):
    ''' This function creates a dataframe that contains the levelized costs and revenues'''

    # 1. Get the row names, except for total_hydrogen_produced
    row_names_levelized_cost = df_discounted_cashflows_for_levelized_cost.index.tolist()
    row_names_levelized_cost.remove('total_electricity_produced')

    # 2. Construct df
    df_levelized_cost = pd.DataFrame(0.0, index=row_names_levelized_cost, columns=['cost','revenues'])
    df_levelized_cost.columns.name = "levelized cost calculations"

    # 3. Calculate total discounted electricity production
    total_electricity_production_sum = df_discounted_cashflows_for_levelized_cost.loc['total_electricity_produced'].sum()

    # 4. Calculate the sum for every row
    row_sums = df_discounted_cashflows_for_levelized_cost.loc[row_names_levelized_cost].sum(axis=1)

    # 5. Arrange data over 'revenues' (if positive) and 'costs' (if negative)
    # use factor 1E6 because cost&revenues are in MEUR
    df_levelized_cost['revenues'] = np.where(row_sums>0, row_sums * 1E6 / total_electricity_production_sum, 0.0)
    df_levelized_cost['cost'] = np.where(row_sums<0, -row_sums * 1E6 / total_electricity_production_sum, 0.0) 

    # 6. Calculate total costs and revenues
    total_costs = df_levelized_cost['cost'].sum()
    total_revenues = df_levelized_cost['revenues'].sum()

    # 7. Calculate profits or unprofitable gap 
        # 7. Calculate profits or unprofitable gap
    df_levelized_cost.loc['profits'] = [0.0, 0.0]
    if total_revenues > total_costs:
        df_levelized_cost.loc['profits', 'cost'] = total_revenues - total_costs

    df_levelized_cost.loc['unprofitable_gap'] = [0.0, 0.0]
    if total_costs > total_revenues: 
        df_levelized_cost.loc['unprofitable_gap', 'revenues'] = total_costs - total_revenues

    return df_levelized_cost


In [47]:
df_levelized_cost_and_revenues = levelized_cost_and_revenues(df_discounted_cashflows_for_levelized_cost,
                                                             **owf_parameters)
df_levelized_cost_and_revenues.style.format(precision=2)

levelized cost calculations,cost,revenues
capex,51.12,0.00
opex,10.57,0.00
decommissioning,0.36,0.00
interest_costs,10.07,0.00
contingency,5.49,0.00
tax_expenses,0.16,0.00
revenues_ppa,0.00,30.85
revenues_market,0.00,3.80
profits,0.00,0.00
unprofitable_gap,0.00,43.11


Project KPI's

In [48]:
from finances.kpis import project_kpi

df_project_kpi = project_kpi(df_present_value_cashflows,
                             df_taxes_and_profits_part2,
                             df_construction_phase,
                             **owf_parameters)

df_project_kpi.style.format(precision=2)

Project KPIs,Value,Unit
net_present_value,-785.63,MEUR
internal_rate_of_return,1.12,%
return_on_investment,-13.08,%
payback_period,-191.14,years
discounted_return_on_investment,-52.89,%
discounted_payback_period,53.06,years


Equity KPI's

In [50]:
from finances.kpis import equity_kpi

df_project_kpi = equity_kpi(df_equity_funding,
                            df_present_value_cashflows,
                             **owf_parameters)

df_project_kpi.style.format(precision=2)

Equity KPIs,Value,Unit
net_present_value,-656.91,MEUR
internal_rate_of_return,-1.98,%
return_of_investment,52.65,%
payback_period,47.48,years
discounted_return_on_investment,-133.51,%
discounted_payback_period,-74.61,years


Output KPI's

In [52]:
from finances.kpis import output_kpi

df_project_kpi = output_kpi(df_levelized_cost_and_revenues,
                            **owf_parameters)

df_project_kpi.style.format(precision=2)

Output KPIs,Value,Unit
levelized_cost,77.77,Eur/MWh
levelized_revenues,34.66,Eur/MWh
levelized_profits,-43.11,Eur/MWh
